# CelebA Wavy_Hair Classifier Training (Colab)

CelebA Wavy_Hair (long-hair proxy) classifier training — for Google Colab.

Drop these cells into a Colab notebook in order. Splits into clearly marked cells with
'# %%' so you can paste each block as its own cell, or just run the whole file as a
script.

This classifier is used to pseudo-label VGGFace2 images as long-hair / short-hair for
the conditional GAN (Task 2), since VGGFace2 itself has no attribute annotations.


In [ ]:
# Mount Drive (optional but recommended) so the CelebA download + checkpoints survive
# across Colab session restarts. Skip this cell if you don't want Drive persistence.

from google.colab import drive
drive.mount('/content/drive')

import os
PERSIST_DIR = "/content/drive/MyDrive/celeba_attr_classifier"  # change if you like
os.makedirs(PERSIST_DIR, exist_ok=True)

In [ ]:
# Environment setup

import os, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as T
import torchvision.utils as vutils
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DATA_ROOT = "/content/celeba_data"   # raw download location (fast local disk)
os.makedirs(DATA_ROOT, exist_ok=True)

In [ ]:
# Download CelebA.
#
# IMPORTANT: torchvision's CelebA(download=True) pulls from CelebA's official Google
# Drive mirror, which frequently hits Google's daily download-quota limit and fails
# with a "Google Drive quota exceeded" RuntimeError — this is a known, long-standing
# issue and is NOT something you can fix from your end; it's rate-limited per-file on
# Google's side. Try option A first; if it fails, use option B (Kaggle mirror).

# --- Option A: official torchvision download ---
try:
    import torchvision.datasets as dsets
    _probe = dsets.CelebA(root=DATA_ROOT, split="train", target_type="attr", download=True)
    print("Official CelebA download succeeded.")
    CELEBA_SOURCE = "torchvision"
except Exception as e:
    print("Official download failed (expected if Google's quota is hit):", str(e)[:300])
    CELEBA_SOURCE = None

# --- Option B: Kaggle mirror via kagglehub (use if Option A failed) ---
if CELEBA_SOURCE is None:
    print("Falling back to Kaggle mirror via kagglehub...")
    # 1. Upload your kaggle.json (Kaggle account -> Settings -> Create New API Token)
    #    Run this in its own cell first if not already done:
    #
    # from google.colab import files
    # files.upload()  # select kaggle.json
    # os.makedirs("/root/.kaggle", exist_ok=True)
    # os.system("cp kaggle.json /root/.kaggle/kaggle.json")
    # os.system("chmod 600 /root/.kaggle/kaggle.json")

    os.system("pip install -q kagglehub")
    import kagglehub
    # jessicali9530/celeba-dataset mirrors img_align_celeba + list_attr_celeba.csv
    kaggle_path = kagglehub.dataset_download("jessicali9530/celeba-dataset")
    print("Kaggle CelebA downloaded to:", kaggle_path)
    CELEBA_SOURCE = "kaggle"
    KAGGLE_CELEBA_PATH = kaggle_path

In [ ]:
# Dataset class. Branches on which source succeeded in Cell 3. Either way, the
# output is the same: a Dataset that yields (image_tensor, wavy_hair_label).

IMG_SIZE = 64
ATTR_NAME = "Wavy_Hair"   # closest CelebA attribute to "long hair"; swap to "Bald" or
                          # build a combined "Straight_Hair"==0 & "Bald"==0 rule if you
                          # want a stricter long-hair proxy

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.CenterCrop(IMG_SIZE),
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3),   # -> [-1, 1], matches TinyAttrClassifier's expected input range
])

eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3),
])

if CELEBA_SOURCE == "torchvision":
    import torchvision.datasets as dsets

    class CelebAWavyHair(torch.utils.data.Dataset):
        """Wraps torchvision's CelebA, extracting just the single Wavy_Hair attribute
        as a binary 0/1 label (torchvision's CelebA already maps -1/1 -> 0/1)."""
        def __init__(self, split, transform):
            self.base = dsets.CelebA(root=DATA_ROOT, split=split, target_type="attr",
                                      download=False, transform=transform)
            self.attr_idx = self.base.attr_names.index(ATTR_NAME)

        def __len__(self):
            return len(self.base)

        def __getitem__(self, idx):
            img, attrs = self.base[idx]
            label = attrs[self.attr_idx].float()
            return img, label

    train_dataset = CelebAWavyHair("train", train_transform)
    valid_dataset = CelebAWavyHair("valid", eval_transform)

elif CELEBA_SOURCE == "kaggle":
    import pandas as pd
    from PIL import Image

    img_dir_candidates = [
        os.path.join(KAGGLE_CELEBA_PATH, "img_align_celeba", "img_align_celeba"),
        os.path.join(KAGGLE_CELEBA_PATH, "img_align_celeba"),
    ]
    IMG_DIR = next(p for p in img_dir_candidates if os.path.isdir(p))
    attr_csv_candidates = [
        os.path.join(KAGGLE_CELEBA_PATH, "list_attr_celeba.csv"),
    ]
    ATTR_CSV = next(p for p in attr_csv_candidates if os.path.isfile(p))
    eval_partition_csv = os.path.join(KAGGLE_CELEBA_PATH, "list_eval_partition.csv")

    attr_df = pd.read_csv(ATTR_CSV)
    attr_df = attr_df.set_index("image_id")
    # Kaggle mirror's CSV already uses -1/1; map to 0/1
    attr_df[ATTR_NAME] = (attr_df[ATTR_NAME] == 1).astype("float32")

    partition_df = pd.read_csv(eval_partition_csv).set_index("image_id")
    # partition: 0 = train, 1 = valid, 2 = test  (matches official CelebA split convention)

    class CelebAWavyHairKaggle(torch.utils.data.Dataset):
        def __init__(self, split, transform):
            split_map = {"train": 0, "valid": 1, "test": 2}
            target = split_map[split]
            ids = partition_df[partition_df["partition"] == target].index.tolist()
            self.ids = [i for i in ids if i in attr_df.index]
            self.transform = transform

        def __len__(self):
            return len(self.ids)

        def __getitem__(self, idx):
            image_id = self.ids[idx]
            img = Image.open(os.path.join(IMG_DIR, image_id)).convert("RGB")
            label = torch.tensor(attr_df.loc[image_id, ATTR_NAME], dtype=torch.float32)
            return self.transform(img), label

    train_dataset = CelebAWavyHairKaggle("train", train_transform)
    valid_dataset = CelebAWavyHairKaggle("valid", eval_transform)

else:
    raise RuntimeError("Neither torchvision nor Kaggle CelebA source succeeded — check Cell 3 output.")

print(f"Train size: {len(train_dataset)} | Valid size: {len(valid_dataset)}")

# Quick class-balance check — Wavy_Hair is imbalanced in CelebA (~32% positive), worth
# knowing before you interpret accuracy numbers.
sample_n = min(5000, len(train_dataset))
pos = sum(int(train_dataset[i][1].item()) for i in range(sample_n))
print(f"Positive rate in first {sample_n} train samples: {pos/sample_n:.3f}")

In [ ]:
# DataLoaders

BATCH_SIZE = 128
NUM_WORKERS = 2  # Colab's default runtime has limited CPU cores; 2-4 is usually the sweet spot

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

print("Batches per epoch (train):", len(train_loader))
print("Batches per epoch (valid):", len(valid_loader))

In [ ]:
# Model (your TinyAttrClassifier, unchanged) + optimizer/loss + checkpoint-resume setup

class TinyAttrClassifier(nn.Module):
    def __init__(self, feat=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, feat, 4, 2, 1), nn.LeakyReLU(0.2, True),         # 64->32
            nn.Conv2d(feat, feat*2, 4, 2, 1), nn.BatchNorm2d(feat*2), nn.LeakyReLU(0.2, True),  # 32->16
            nn.Conv2d(feat*2, feat*4, 4, 2, 1), nn.BatchNorm2d(feat*4), nn.LeakyReLU(0.2, True), # 16->8
            nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Linear(feat*4, 1)
    def forward(self, x):
        h = self.net(x).flatten(1)
        return self.fc(h).squeeze(1)   # logit, sigmoid -> P(long hair)

attr_clf = TinyAttrClassifier().to(device)
print("Params:", sum(p.numel() for p in attr_clf.parameters()))

# Class imbalance handling: weight the positive class up since Wavy_Hair is the
# minority class (~32% positive). Without this the classifier tends to just predict
# "not wavy" and still get decent accuracy while being useless for pseudo-labeling.
pos_weight = torch.tensor([ (1 - 0.32) / 0.32 ], device=device)  # ~2.1x — adjust after
                                                                  # checking the actual
                                                                  # positive rate printed in Cell 4
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(attr_clf.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1)

CKPT_PATH = os.path.join(PERSIST_DIR, "attr_clf_latest.pt")
NUM_EPOCHS = 10

start_epoch = 0
best_val_loss = float("inf")
history = {"train_loss": [], "val_loss": [], "val_acc": []}

if os.path.exists(CKPT_PATH):
    print("Resuming from checkpoint...")
    ckpt = torch.load(CKPT_PATH, map_location=device)
    attr_clf.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_epoch = ckpt["epoch"]
    best_val_loss = ckpt.get("best_val_loss", float("inf"))
    history = ckpt.get("history", history)
    print(f"Resumed at epoch {start_epoch}, best_val_loss={best_val_loss:.4f}")

In [ ]:
# Train + validate loop with checkpointing every epoch and a separate "best" checkpoint

def evaluate(loader):
    attr_clf.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = attr_clf(imgs)
            loss = criterion(logits, labels)
            total_loss += loss.item() * imgs.size(0)
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += imgs.size(0)
    attr_clf.train()
    return total_loss / total, correct / total

for epoch in range(start_epoch, NUM_EPOCHS):
    t0 = time.time()
    attr_clf.train()
    running_loss, running_n = 0.0, 0

    for i, (imgs, labels) in enumerate(train_loader):
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = attr_clf(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        running_n += imgs.size(0)

        if (i + 1) % 200 == 0:
            print(f"epoch {epoch} step {i+1}/{len(train_loader)} | "
                  f"running train loss {running_loss/running_n:.4f}")

    train_loss = running_loss / running_n
    val_loss, val_acc = evaluate(valid_loader)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"[epoch {epoch}] train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
          f"val_acc={val_acc:.4f}  ({time.time()-t0:.1f}s)")

    # Always save "latest" so resume works
    torch.save({
        "model": attr_clf.state_dict(),
        "optimizer": optimizer.state_dict(),
        "epoch": epoch + 1,
        "best_val_loss": best_val_loss,
        "history": history,
    }, CKPT_PATH)

    # Separately save "best" by validation loss — this is the checkpoint you actually
    # want to use for pseudo-labeling VGGFace2, not necessarily the last epoch
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(attr_clf.state_dict(), os.path.join(PERSIST_DIR, "attr_clf_best.pt"))
        print(f"  -> new best (val_loss={val_loss:.4f}), saved attr_clf_best.pt")

print("Training finished (or interrupted — re-run this cell to resume from the last epoch).")

In [ ]:
# Plot training curves — use this figure in your report's "Experimental Setup" /
# "Results" section to show the attribute classifier converged before you trust its
# pseudo-labels.

epochs_range = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs_range, history["train_loss"], label="train loss")
axes[0].plot(epochs_range, history["val_loss"], label="val loss")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("BCE loss"); axes[0].legend(); axes[0].set_title("Loss")
axes[1].plot(epochs_range, history["val_acc"], color="green")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].set_title("Validation accuracy")
plt.tight_layout()
plt.savefig(os.path.join(PERSIST_DIR, "training_curves.png"), dpi=150)
plt.show()

In [ ]:
# Sanity-check the trained classifier visually: show a grid of validation images with
# predicted vs true label, so you can eyeball whether "long hair" predictions look right
# before trusting it to pseudo-label all of VGGFace2.

attr_clf.eval()
sample_imgs, sample_labels = next(iter(valid_loader))
sample_imgs_gpu = sample_imgs.to(device)
with torch.no_grad():
    probs = torch.sigmoid(attr_clf(sample_imgs_gpu)).cpu()

n_show = 16
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    img = sample_imgs[i].permute(1, 2, 0).numpy()
    img = (img * 0.5 + 0.5).clip(0, 1)  # undo normalization for display
    ax.imshow(img)
    true_label = int(sample_labels[i].item())
    pred_label = int(probs[i].item() > 0.5)
    pred_prob = probs[i].item()
    color = "green" if true_label == pred_label else "red"
    ax.set_title(f"true={true_label} pred={pred_label} (p={pred_prob:.2f})",
                 color=color, fontsize=9)
    ax.axis("off")
plt.suptitle("Wavy_Hair classifier — validation predictions (green=correct, red=wrong)")
plt.tight_layout()
plt.savefig(os.path.join(PERSIST_DIR, "validation_predictions_grid.png"), dpi=150)
plt.show()
attr_clf.train()

print(f"\nBest checkpoint saved at: {os.path.join(PERSIST_DIR, 'attr_clf_best.pt')}")
print("Download this .pt file (or keep it in Drive) and load it in the main GAN notebook")
print("via: attr_clf.load_state_dict(torch.load('attr_clf_best.pt')) for VGGFace2 pseudo-labeling.")